# Part 2 – Python: Building a Reproducible Traffic Analytics Pipeline

## Task 2 – Feature Engineering (NumPy and Pandas)

This section transforms the cleaned traffic dataset into ML-ready features
using Pandas and NumPy.

Task 2 includes:

1. Creating time-based features: hour, day of week, and weekend indicator
2. Creating cyclical time encodings
3. Encoding relevant categorical weather variables
4. Creating useful derived weather indicators
5. Scaling or normalising continuous numerical variables
6. Creating a data-driven congestion category from traffic volume
7. Logging dataset shapes and intermediate feature-engineering values

### Task 2.1 – Load and Inspect the Cleaned Dataset

The cleaned dataset produced by Task 1 is loaded as the starting point for
feature engineering. Its shape and data types are inspected before any new
features are created.

In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
cleaned_data_path = Path("cleaned_traffic_data.csv")

traffic_df = pd.read_csv(
    cleaned_data_path,
    parse_dates=["date_time"]
)

traffic_df.shape

(48187, 9)

In [3]:
traffic_df["date_time"].dtype

dtype('<M8[ns]')

In [4]:
traffic_df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


**Result:** The cleaned dataset was loaded successfully with 48,187 rows and
9 columns. The `date_time` column was correctly parsed as a Pandas datetime
data type, providing the required foundation for time-based feature engineering.

### Task 2.2 – Configure Feature Engineering Logging

Logging is configured to record the feature-engineering workflow. INFO messages
document normal processing milestones and dataset shapes, while DEBUG messages
record intermediate calculations such as thresholds used to construct derived
features.

In [5]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
    )

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        "feature_engineering.log",
        mode="a"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

In [6]:
logger.info(
    "Feature engineering started: %d rows, %d columns.",
    traffic_df.shape[0],
    traffic_df.shape[1]
)

2026-10-06 07:40:16,848 - INFO - __main__ - Feature engineering started: 48187 rows, 9 columns.


### Task 2.3 – Create Time-Based Features

Time-based features are derived from the validated `date_time` column to capture
daily and weekly traffic patterns.

The following features are created:

- `hour`: hour of the day from 0 to 23
- `day_of_week`: day of the week from Monday = 0 to Sunday = 6
- `is_weekend`: binary indicator where Saturday and Sunday = 1
- `hour_sin` and `hour_cos`: cyclical encodings that preserve the circular
  relationship between hour 23 and hour 0

In [7]:
def create_time_features(df):
    """Create time-based and cyclical features from date_time."""
    df = df.copy()

    df["hour"] = df["date_time"].dt.hour
    df["day_of_week"] = df["date_time"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    logger.info(
        "Created time features: hour, day_of_week, is_weekend, "
        "hour_sin, and hour_cos."
    )

    return df

In [8]:
traffic_df = create_time_features(traffic_df)

traffic_df.shape

2026-10-06 07:40:16,894 - INFO - __main__ - Created time features: hour, day_of_week, is_weekend, hour_sin, and hour_cos.


(48187, 14)

In [9]:
traffic_df[
    [
        "date_time",
        "hour",
        "day_of_week",
        "is_weekend",
        "hour_sin",
        "hour_cos"
    ]
].head(10)

,date_time,hour,day_of_week,is_weekend,hour_sin,hour_cos
0,2012-10-02 09:00:00,9,1,0,7.071068e-01,-7.071068e-01
1,2012-10-02 10:00:00,10,1,0,5.000000e-01,-8.660254e-01
2,2012-10-02 11:00:00,11,1,0,2.588190e-01,-9.659258e-01
3,2012-10-02 12:00:00,12,1,0,1.224647e-16,-1.000000e+00
4,2012-10-02 13:00:00,13,1,0,-2.588190e-01,-9.659258e-01
5,2012-10-02 14:00:00,14,1,0,-5.000000e-01,-8.660254e-01
6,2012-10-02 15:00:00,15,1,0,-7.071068e-01,-7.071068e-01
7,2012-10-02 16:00:00,16,1,0,-8.660254e-01,-5.000000e-01
8,2012-10-02 17:00:00,17,1,0,-9.659258e-01,-2.588190e-01
9,2012-10-02 18:00:00,18,1,0,-1.000000e+00,-1.836970e-16


### Task 2.4 – Create Weather Features

Weather-related features are engineered to make categorical and continuous
weather information more suitable for machine-learning applications.

`weather_main` is encoded using one-hot encoding because it represents
nominal weather categories with no natural ordering.

Additional binary indicators are derived from the weather measurements to
capture whether precipitation or snow was recorded during each observation.

In [10]:
def create_weather_features(df):
    """Create encoded and derived weather features."""
    df = df.copy()

    # Derived binary weather indicators
    df["has_rain"] = (df["rain_1h"] > 0).astype(int)
    df["has_snow"] = (df["snow_1h"] > 0).astype(int)

    # One-hot encode the main weather category
    weather_dummies = pd.get_dummies(
        df["weather_main"],
        prefix="weather",
        dtype=int
    )

    df = pd.concat(
        [df, weather_dummies],
        axis=1
    )

    logger.info(
        "Created weather features: has_rain, has_snow, and %d "
        "one-hot encoded weather_main columns.",
        weather_dummies.shape[1]
    )

    return df

In [11]:
traffic_df = create_weather_features(traffic_df)

traffic_df.shape

2026-10-06 07:40:16,983 - INFO - __main__ - Created weather features: has_rain, has_snow, and 11 one-hot encoded weather_main columns.


(48187, 27)

In [12]:
weather_feature_columns = [
    column for column in traffic_df.columns
    if column.startswith("weather_")
]

weather_feature_columns

['weather_main',
 'weather_description',
 'weather_Clear',
 'weather_Clouds',
 'weather_Drizzle',
 'weather_Fog',
 'weather_Haze',
 'weather_Mist',
 'weather_Rain',
 'weather_Smoke',
 'weather_Snow',
 'weather_Squall',
 'weather_Thunderstorm']

### Task 2.5 – Scale Continuous Numerical Features

Continuous numerical variables are transformed using Min-Max scaling so that
their values fall between 0 and 1.

The `temp` and `clouds_all` variables are scaled while the original columns are
retained. Keeping both the original and scaled versions supports interpretation
and later machine-learning workflows.

In [13]:
def create_scaled_features(df):
    """Create Min-Max scaled versions of selected continuous variables."""
    df = df.copy()

    columns_to_scale = ["temp", "clouds_all"]

    for column in columns_to_scale:
        column_min = df[column].min()
        column_max = df[column].max()

        logger.debug(
            "%s scaling range: min=%.2f, max=%.2f.",
            column,
            column_min,
            column_max
        )

        if column_max == column_min:
            df[f"{column}_scaled"] = 0.0

            logger.warning(
                "Column '%s' has no variation; scaled values set to 0.",
                column
            )
        else:
            df[f"{column}_scaled"] = (
                (df[column] - column_min)
                / (column_max - column_min)
            )

    logger.info(
        "Created Min-Max scaled features for: %s.",
        ", ".join(columns_to_scale)
    )

    return df

In [14]:
traffic_df.shape

(48187, 27)

In [15]:
[column for column in traffic_df.columns if column.startswith("weather_")]

['weather_main',
 'weather_description',
 'weather_Clear',
 'weather_Clouds',
 'weather_Drizzle',
 'weather_Fog',
 'weather_Haze',
 'weather_Mist',
 'weather_Rain',
 'weather_Smoke',
 'weather_Snow',
 'weather_Squall',
 'weather_Thunderstorm']

In [16]:
traffic_df["weather_main"].nunique()

11

In [17]:
traffic_df.shape


(48187, 27)

In [18]:
traffic_df = create_scaled_features(traffic_df)

traffic_df.shape

2026-10-06 07:40:17,099 - INFO - __main__ - Created Min-Max scaled features for: temp, clouds_all.


(48187, 29)

### Task 2.6 – Create a Data-Driven Congestion Category

A categorical congestion target is derived from `traffic_volume` using the
observed distribution of traffic demand rather than arbitrary fixed thresholds.

The 25th and 75th percentiles are used as data-driven boundaries:

- `Low`: traffic volume at or below the 25th percentile
- `Medium`: traffic volume between the 25th and 75th percentiles
- `High`: traffic volume above the 75th percentile

The calculated thresholds are logged at DEBUG level so that the target
construction remains traceable and reproducible.

In [19]:
q1 = traffic_df["traffic_volume"].quantile(0.25)
q3 = traffic_df["traffic_volume"].quantile(0.75)

logger.debug(
    "Congestion thresholds calculated: Q1=%.2f, Q3=%.2f.",
    q1,
    q3
)

q1, q3

(np.float64(1192.5), np.float64(4933.0))

In [20]:
def create_congestion_category(df):
    """Create a data-driven congestion category using traffic-volume quartiles."""
    df = df.copy()

    q1 = df["traffic_volume"].quantile(0.25)
    q3 = df["traffic_volume"].quantile(0.75)

    logger.debug(
        "Congestion thresholds: Q1=%.2f, Q3=%.2f.",
        q1,
        q3
    )

    conditions = [
        df["traffic_volume"] <= q1,
        df["traffic_volume"] > q3
    ]

    choices = [
        "Low",
        "High"
    ]

    df["congestion_category"] = np.select(
        conditions,
        choices,
        default="Medium"
    )

    logger.info(
        "Created congestion_category using traffic-volume quartiles."
    )

    return df

In [21]:
traffic_df = create_congestion_category(traffic_df)

traffic_df[
    ["traffic_volume", "congestion_category"]
].head(10)

2026-10-06 07:40:17,169 - INFO - __main__ - Created congestion_category using traffic-volume quartiles.


,traffic_volume,congestion_category
0,5545,High
1,4516,Medium
2,4767,Medium
3,5026,High
4,4918,Medium
5,5181,High
6,5584,High
7,6015,High
8,5791,High
9,4770,Medium


In [22]:
traffic_df["congestion_category"].value_counts()

congestion_category
Medium    24103
Low       12047
High      12037
Name: count, dtype: int64

**Result:** The congestion target was created using data-driven quartile
thresholds derived from `traffic_volume`.

- Q1 (25th percentile): 1,192.5 vehicles
- Q3 (75th percentile): 4,933.0 vehicles
- Low congestion: 12,047 observations
- Medium congestion: 24,103 observations
- High congestion: 12,037 observations

The resulting distribution is approximately 25% Low, 50% Medium, and 25% High,
providing a balanced and reproducible congestion classification based on the
observed traffic-volume distribution.


### Task 2.7 – Final Feature Validation and Save Engineered Data

The engineered dataset is validated to confirm that all required ML-ready
features were created successfully and contain no unexpected missing values.

The final dataset is then saved separately so that the cleaned Task 1 dataset
remains unchanged and can be reproduced independently.

In [23]:
required_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "has_rain",
    "has_snow",
    "temp_scaled",
    "clouds_all_scaled",
    "congestion_category"
]

missing_features = [
    feature for feature in required_features
    if feature not in traffic_df.columns
]

missing_features

[]

In [24]:
traffic_df[required_features].isna().sum()

hour                   0
day_of_week            0
is_weekend             0
hour_sin               0
hour_cos               0
has_rain               0
has_snow               0
temp_scaled            0
clouds_all_scaled      0
congestion_category    0
dtype: int64

In [25]:
traffic_df.shape

(48187, 30)

In [26]:
logger.info(
    "Feature engineering completed: %d rows, %d columns.",
    traffic_df.shape[0],
    traffic_df.shape[1]
)

2026-10-06 07:40:17,271 - INFO - __main__ - Feature engineering completed: 48187 rows, 30 columns.


In [27]:
engineered_output_path = Path("engineered_traffic_data.csv")

try:
    traffic_df.to_csv(
        engineered_output_path,
        index=False
    )

    logger.info(
        "Engineered dataset saved successfully to '%s': %d rows, %d columns.",
        engineered_output_path,
        traffic_df.shape[0],
        traffic_df.shape[1]
    )

except OSError:
    logger.error(
        "Failed to save engineered dataset to '%s'.",
        engineered_output_path,
        exc_info=True
    )

2026-10-06 07:40:18,296 - INFO - __main__ - Engineered dataset saved successfully to 'engineered_traffic_data.csv': 48187 rows, 30 columns.


In [28]:
engineered_output_path.exists()

True

**Task 2 Result:** Feature engineering was completed successfully. The final
engineered dataset contains 48,187 rows and 30 columns.

The workflow created time-based features, cyclical hour encodings, encoded
weather features, precipitation indicators, Min-Max scaled numerical features,
and a data-driven congestion target based on traffic-volume quartiles.

All required engineered features were validated successfully with no unexpected
missing values. The resulting dataset was saved as
`engineered_traffic_data.csv` for use in subsequent analysis and machine-learning
tasks.